In [ ]:
import csv
import random
import re

In [ ]:
# Исходные данные (вопросы и ответы)
base_qa = [
("Какие есть формы обучения в колледже?","В колледже представлены две формы обучения: очная и заочная. Для некоторых специальностей также упоминается очно-заочная форма с применением дистанционных образовательных технологий (ДОТ) — информация о ней указана в таблице стоимости, но сроки в предоставленных данных для неё не приведены."),
("Что означает «база основного общего образования»?","Это обучение на базе 9 классов (общее образование)."),
("Что означает «база среднего (полного) общего образования»?","Это обучение на базе 11 классов (среднее общее образование)."),
("Могу ли я учиться быстрее стандартного срока?","Да, возможно ускоренное обучение. Оно регламентируется Приказом Минобрнауки №464 (ст.24). Право на ускоренное обучение имеют лица, уже имеющие квалификацию по профессии СПО, соответствующей выбранной специальности. Порядок устанавливается локальными актами образовательной организации."),
("Обучение по индивидуальному учебному плану доступно в колледже?","Да, образовательная организация может изменять сроки получения образования с учетом особенностей и образовательных потребностей конкретного обучающегося, в том числе для ускоренного обучения."),
("Сколько учиться на бухгалтера очно после 9 классов?","2 года 10 месяцев. Специальность 38.02.01 «Экономика и бухгалтерский учет»."),
("А после 11 классов на бухгалтера очно?","1 год 10 месяцев."),
("Какой срок обучения на программиста очно после 9 класса?","3 года 10 месяцев. Специальность 09.02.11 «Разработка и управление программным обеспечением»."),
("Есть ли специальность «Дизайн», и сколько учиться после 11 класса?","Да, специальность 54.02.01 «Дизайн (по отраслям)». Срок обучения на базе 11 классов – 2 года 10 месяцев."),
("Сколько учиться на юриста очно после 9 классов?","2 года 10 месяцев. Доступны две квалификации: «Юрист в сфере социального обеспечения» и «Юрист в сфере судебного администрирования» (код 40.02.04)."),
("Какая специальность имеет самый долгий срок обучения на базе 9 классов?","Несколько специальностей требуют 3 года 10 месяцев: «Компьютерные системы и комплексы», «Разработка и управление ПО», «Дошкольное образование», «Физическая культура», «Дизайн», «Интеграция решений с применением технологий ИИ»."),
("Есть ли специальности, которые доступны только с 11 классами (не принимают после 9)?","Да, согласно таблице: «Преподавание в начальных классах» (только 2 года 10 месяцев на базе 11 классов), «Обеспечение информационной безопасности автоматизированных систем» (только 2 года 10 месяцев), «Технологии индустрии красоты» (только 1 год 10 месяцев), «Техническое обслуживание и ремонт автотранспортных средств» (только 2 года 10 месяцев)."),
("Сколько учиться на веб-разработчика после 11 классов очно?","1 год 10 месяцев. Специальность 09.02.09 «Веб-разработка»."),
("Можно ли выучиться на банковского специалиста заочно?","Да, специальность 38.02.07 «Банковское дело». Сроки: после 9 классов – 3 года 4 месяца, после 11 классов – 2 года 4 месяца."),
("Какая специальность имеет максимальный срок заочного обучения после 9 классов?","4 года 4 месяца. Это специальности: 44.02.01 «Дошкольное образование» и 09.02.11 «Разработка и управление программным обеспечением»."),
("Можно ли заочно учиться на финансиста после 11 классов?","Да, специальность 38.02.06 «Финансы». Срок – 2 года 4 месяца."),
("Доступна ли заочная форма для специальности «Реклама»?","Да, 42.02.01 «Реклама». Срок: после 9 классов – 3 года 4 месяца, после 11 классов – 2 года 4 месяца."),
("Сколько стоит семестр на очном отделении «Банковское дело»?","100 000 рублей на базе 9 классов, 110 000 рублей на базе 11 классов."),
("В каких специальностях очная форма стоит 90 000–100 000 рублей на базе 9 классов?","«Дошкольное образование» (90 000), «Физическая культура» (90 000), «Финансы» (90 000), а также «Конгрессно-выставочная деятельность» (80 000) — это дешевле. «Дизайн» и «Банковское дело» — по 100 000."),
("Какая самая дорогая очная специальность на базе 11 классов?","120 000 рублей за семестр. Это: «Веб-разработка», «Интеграция с ИИ», «Компьютерные системы», «Разработка ПО», а также «Техническая эксплуатация ИС» (указано 120 000)."),
("Есть ли специальности с очной стоимостью 80 000 рублей на базе 9 классов?","Да, две: «Конгрессно-выставочная деятельность» (43.02.16) и «Техническая эксплуатация и сопровождение ИС» (код 09.02.06 в таблице стоимости)."),
("Сколько стоит заочное обучение (с ДОТ) за семестр?","По предоставленной таблице, заочная форма с ДОТ на базе и 9, и 11 классов стоит 50 000 рублей за семестр для указанных специальностей: Банковское дело, Дошкольное образование, Операционная деятельность в логистике, Разработка ПО, Реклама, Техническая эксплуатация ИС, Торговое дело, Финансы."),
("Указана ли стоимость для очно-заочной формы обучения?","В таблице есть столбцы «ОЧНО-ЗАОЧНАЯ (с применением ДОТ)», но все ячейки для неё пусты. Стоимость по этой форме не предоставлена."),
("В специальности «Экономика и бухгалтерский учет» есть заочная форма по указанной цене?","Нет. В таблице стоимости для 38.02.01 указана только очная форма (100 000 / 110 000). Заочная форма для этой специальности в таблице стоимости отсутствует."),
("Какие документы нужны для поступления в колледж?","1. Аттестат; 2. Паспорт; 3. Медицинская справка 086-у; 4. Ксерокопия приписного свидетельства (для юношей с 17 лет); 5. 2 фотографии 3×4; 6. Паспорт родителя (если поступающему нет 18 лет)."),
("Как можно подать документы?","Тремя способами: 1. Лично в университете (Москва, 2-й Кожуховский проезд, 12, стр.1); 2. По почте; 3. В электронной форме через ЕПГУ («Госуслуги»)."),
("Можно ли подать заявление через «Госуслуги»?","Да. В личном кабинете на портале Госуслуги можно заполнить заявку, загрузить документы и потом подтвердить поступление."),
("Адрес и телефон приёмной комиссии?","Адрес: 115432, Москва, 2-й Кожуховский проезд, д.12, стр.1, каб. 101. Телефон: 8 (495) 818-99-43. E-mail: pk@muiv.ru"),
("Когда работает приёмная комиссия?","В период с 23 сентября 2025 года по 19 июня 2026 года. График: понедельник–четверг с 09:30 до 18:15, пятница с 09:30 до 17:00. Суббота и воскресенье – выходной."),
("Может ли доверенное лицо подать документы за меня?","Да, лично или через доверенное лицо в здании университета."),
("По какой специальности готовят «Специалиста по работе с искусственным интеллектом»?","Код 09.02.13 «Интеграция решений с применением технологий искусственного интеллекта». Срок обучения очно: после 9 классов – 3 года 10 месяцев, после 11 – 2 года 10 месяцев."),
("Есть ли специальность «Физическая культура» и сколько стоит?","Да, 49.02.01. Очная форма: 90 000 руб. (9 кл.) / 100 000 руб. (11 кл.). Срок: 3 г.10 м / 2 г.10 м."),
("Можно ли получить специальность «Техническое обслуживание автотранспорта» заочно?","В таблице сроков заочной формы эта специальность (23.02.07) не указана. В таблице стоимости она есть только в очной (на базе СОО — 80 000 руб.). Скорее всего, заочной формы для неё нет."),
("Квалификация «Бухгалтер» — это какая специальность?","38.02.01 «Экономика и бухгалтерский учет». Форма обучения — очная (заочная в таблице сроков отсутствует)."),
("Что означает прочерк «-» в таблице сроков?","Прочерк означает, что обучение по данной специальности на указанной базе (например, 9 классов) не ведётся. Пример: «Технологии индустрии красоты» — только с 11 классами."),
("Сколько учиться на дошкольное образование очно после 9 классов?","3 года 10 месяцев. Специальность 44.02.01 «Дошкольное образование», квалификация «Воспитатель детей дошкольного возраста в полилингвальной образовательной среде»."),
("Какие специальности на заочной форме имеют срок 3 года 4 месяца после 9 классов?","«Операционная деятельность в логистике», «Торговое дело», «Банковское дело», «Реклама», «Финансы», «Техническая эксплуатация и сопровождение информационных систем»."),
("Сколько стоит очное обучение на специальности «Юриспруденция»?","100 000 рублей за семестр на базе 9 классов и 110 000 рублей на базе 11 классов."),
("Есть ли специальность «Конгрессно-выставочная деятельность»?","Да, код 38.02.09. Очная форма: срок 2 года 10 месяцев (9 кл.) / 1 год 10 месяцев (11 кл.), стоимость 80 000 (9 кл.) / 90 000 (11 кл.). Заочная форма в таблице стоимости не указана."),
("Что за специальность 09.02.12?","«Техническая эксплуатация и сопровождение информационных систем». Срок очно: 2 года 10 месяцев (9 кл.) / 1 год 10 месяцев (11 кл.). Заочно: 3 года 4 месяца (9 кл.) / 2 года 4 месяца (11 кл.)."),
("Какая квалификация у выпускников по специальности «Торговое дело»?","Специалист торгового дела."),
("Можно ли заочно учиться на программиста после 11 классов?","Да, 09.02.11 «Разработка и управление программным обеспечением». Срок – 3 года 4 месяца, стоимость заочного обучения с ДОТ – 50 000 рублей за семестр."),
]

In [ ]:
# Блок бакалавриат/специалитет (также около 40 пар)
more_qa = [
("Что такое бакалавриат?","Бакалавриат — первая ступень высшего образования. Традиционный срок обучения — 4 года (есть вариации). Выпускник получает диплом государственного образца и может работать или поступать в магистратуру."),
("Сколько длится очное обучение в бакалавриате?","Обычно 4 года. Исключение: 44.03.05 «Педагогическое образование (с двумя профилями подготовки)» — 5 лет очно."),
("Сколько длится очно-заочное (заочное) обучение в бакалавриате?","4 года 6 месяцев. Для группы выходного дня — также 4 года 6 месяцев (кроме 38.03.05 «Бизнес-информатика» с профилем «Игровая компьютерная индустрия» — там 3 года 6 месяцев)."),
("Можно ли учиться в бакалавриате ускоренно на базе СПО?","Да. На базе среднего профессионального образования при наличии квалификации по специальности сроки: очная — 3 года, очно-заочная дистанционная — 3 года 2 месяца, заочная группа выходного дня — 3 года 6 месяцев."),
("Что такое специалитет?","Специалитет — традиционная для России ступень высшего образования. Срок: 5 лет (очная форма) или 6 лет (очно-заочная / заочная). Выдаётся диплом специалиста, после него можно поступать в магистратуру и аспирантуру."),
("Есть ли ускоренное обучение в бакалавриате для тех, у кого уже есть высшее образование?","Да. На базе высшего образования возможна ускоренная очно-заочная классическая форма — 3 года 2 месяца, а также заочная дистанционная — 3 года. В некоторых программах присутствует группа выходного дня."),
("Сколько стоит бакалавриат по направлению «Бизнес-информатика» (очное)?","Для профилей «Игровая компьютерная индустрия», «Цифровой дизайн и веб-разработка», «Бизнес-аналитик 1С» — 166 000 рублей за семестр (на базе СОО и СПО)."),
("Сколько стоит очное обучение на «Менеджменте» в бакалавриате?","166 000 рублей за семестр на базе СОО и СПО. Логистика, маркетинг, управление проектами, цифровой менеджмент — все по 166 000."),
("Сколько стоит бакалавриат по «Прикладной информатике» (ИИ и анализ данных) очно?","150 000 рублей за семестр (на базе СОО и СПО)."),
("Есть ли недорогие направления бакалавриата очно?","Да. «Специальное (дефектологическое) образование» — 110 000 руб./семестр, «Психолого-педагогическое образование» — 130 000 руб., «Туризм» — 130 000 руб., «Педагогическое образование» — 130 000 руб."),
("Сколько стоит заочная форма в бакалавриате с применением ДОТ?","50 000 рублей за семестр (на базе СОО) или 50 000 (на базе СПО / ВО) в зависимости от направления. Примеры: «Бизнес-информатика», «Менеджмент», «Экономика», «Юриспруденция», «Реклама»."),
("Сколько стоит очно-заочная (группа выходного дня) в бакалавриате?","Обычно 100 000–166 000 руб./семестр. Например: «Менеджмент» (управление проектами) — 100 000 руб. на базе СОО, «Экономика» — 100 000 руб. Точная стоимость зависит от направления."),
("Какая стоимость семестра на «Юриспруденции» (гражданско-правовой профиль) очно?","166 000 рублей (на базе СОО и СПО)."),
("Сколько стоит бакалавриат по «Государственному и муниципальному управлению» очно?","166 000 рублей за семестр (на базе СОО и СПО)."),
("Есть ли бакалавриат по «Рекламе»?","Да, код 42.03.01 «Реклама и связи с общественностью». Очная форма — 166 000 руб./семестр. Заочная с ДОТ — 50 000 руб."),
("Сколько стоит бакалавриат по «Экономике» (бизнес-аналитика, бухучет, финансовая аналитика) очно?","166 000 рублей за семестр на базе СОО и СПО."),
("Что такое «группа выходного дня» в бакалавриате?","Это очно-заочная форма обучения с занятиями по выходным дням. Срок обычно 4 года 6 месяцев, но для «Бизнес-информатики» (игровая компьютерная индустрия) — 3 года 6 месяцев. Стоимость зависит от программы."),
("Какие сроки обучения в специалитете очно?","5 лет (нормативный срок для всех программ)."),
("Сколько стоит специалитет по направлению «Таможенное дело» очно?","166 000 рублей за семестр."),
("Есть ли специалитет с заочной формой?","Да. Например, «Таможенное дело» — заочная форма с ДОТ стоит 50 000 руб./семестр. Срок обучения — 6 лет."),
("Что означает «на базе среднего профессионального образования (квалификация по специальности)»?","Это поступление в бакалавриат после колледжа, если у вас уже есть диплом СПО именно по профильной специальности. Для таких студентов возможны ускоренные сроки (очная — 3 года, очно-заочная — 3 года 2 месяца и т.д.)."),
("Можно ли после колледжа поступить на бакалавриат с сокращённым сроком?","Да, если ваша квалификация по профессии/специальности соответствует направлению бакалавриата. Сроки: очно — 3 года, очно-заочная дистанционная — 3 года 2 месяца, заочная группа выходного дня — 3 года 6 месяцев."),
("Сколько стоит бакалавриат по «Педагогическому образованию» с двумя профилями очно?","130 000 рублей за семестр (на базе СОО)."),
("Какова стоимость очного бакалавриата по «Туризму»?","130 000 рублей за семестр."),
("Есть ли бакалавриат по «Прикладной информатике» с профилем «Кибербезопасность цифрового предприятия»?","Да, код 09.03.03. Очная форма — 150 000 руб./семестр. Заочная с ДОТ — 50 000 руб."),
("Сколько стоит бакалавриат по «Психолого-педагогическому образованию» очно?","130 000 рублей за семестр (на базе СОО и СПО)."),
("Какие направления бакалавриата стоят 110 000 рублей за семестр очно?","«Специальное (дефектологическое) образование» (44.03.03), «Регионоведение России» (41.03.02), «Градостроительство» (07.03.04), «Инноватика» (27.03.05), «Техносферная безопасность» (20.03.01), «Математическое обеспечение и администрирование ИС» (02.03.03)."),
("Что означает «индивидуальный учебный план» в бакалавриате?","Это план, который университет разрабатывает под конкретного студента. Позволяет освоить программу ускоренно или углублённо, с учётом его возможностей, потребностей и предшествующего образования."),
("Какой срок обучения в бакалавриате на базе высшего образования?","Для ускоренного обучения: очно-заочная классическая — 3 года 2 месяца, заочная дистанционная — 3 года. В некоторых программах есть заочная группа выходного дня — 3 года 6 месяцев."),
]

In [ ]:
# Объединяем все пары
all_pairs = base_qa + more_qa

In [ ]:
# Функции для генерации парафраз вопроса
def paraphrase_question(q, a):
    """Генерирует 1-3 вариации вопроса на основе шаблонов."""
    variants = []

    # Шаблоны для перефразирования (ключевые слова заменяются)
    # Для простоты используем ручные правила, но можно расширить.

    # 1. Замена "сколько учиться" -> "какова продолжительность" и т.п.
    if re.search(r'сколько учиться', q, re.IGNORECASE):
        v1 = re.sub(r'Сколько учиться', 'Какова продолжительность обучения', q, flags=re.IGNORECASE)
        v2 = re.sub(r'Сколько учиться', 'Какие сроки', q, flags=re.IGNORECASE)
        variants.extend([v1, v2])

    # 2. Замена "сколько стоит" -> "какова стоимость", "назовите цену"
    if re.search(r'сколько стоит', q, re.IGNORECASE):
        v1 = re.sub(r'Сколько стоит', 'Какова стоимость', q, flags=re.IGNORECASE)
        v2 = re.sub(r'Сколько стоит', 'Укажите цену', q, flags=re.IGNORECASE)
        v3 = re.sub(r'Сколько стоит', 'Назовите стоимость обучения', q, flags=re.IGNORECASE)
        variants.extend([v1, v2, v3])

    # 3. Вопрос "что такое ..." -> "дайте определение", "объясните"
    if re.search(r'Что такое', q):
        v1 = re.sub(r'Что такое', 'Дайте определение понятию', q)
        v2 = re.sub(r'Что такое', 'Объясните, что означает', q)
        variants.extend([v1, v2])

    # 4. Вопрос "можно ли" -> "разрешено ли", "есть ли возможность"
    if re.search(r'Можно ли', q):
        v1 = re.sub(r'Можно ли', 'Разрешено ли', q)
        v2 = re.sub(r'Можно ли', 'Существует ли возможность', q)
        v3 = re.sub(r'Можно ли', 'Возможно ли', q)
        variants.extend([v1, v2, v3])

    # 5. Вопрос "какой срок" -> "сколько времени", "какая длительность"
    if re.search(r'Какой срок', q):
        v1 = re.sub(r'Какой срок', 'Сколько времени', q)
        v2 = re.sub(r'Какой срок', 'Какая длительность', q)
        variants.extend([v1, v2])

    # 6. Добавить в начало "Подскажите, " или "Скажите, пожалуйста, "
    polite_v = f"Скажите, пожалуйста, {q[0].lower() + q[1:]}" if q[0].isupper() else f"Скажите, пожалуйста, {q}"
    variants.append(polite_v)

    # 7. Изменить порядок: "после 9 классов" -> "на базе 9 классов" и т.д.
    q2 = q.replace("после 9 классов", "на базе 9 классов").replace("после 11 классов", "на базе 11 классов")
    if q2 != q:
        variants.append(q2)

    # Убираем дубликаты и очищаем
    unique_vars = []
    for v in variants:
        v = v.strip().replace('  ', ' ')
        if v not in unique_vars and v != q:
            unique_vars.append(v)

    return unique_vars[:3]  # не более 3 вариаций на один вызов

In [ ]:
# Генерация итогового датасета с нужным количеством строк
TARGET_ROWS = 5000 # можно менять

In [ ]:
dataset = []

# Для каждой исходной пары генерируем парафразы, пока не наберём нужное количество
# Сохраняем оригинальные вопросы тоже.
for q, a in all_pairs:
    dataset.append((q, a))  # оригинал

# Генерируем дополнительные варианты
additional_needed = TARGET_ROWS - len(dataset)
if additional_needed > 0:
    # Циклически проходим по парам и генерируем вариации
    idx = 0
    while len(dataset) < TARGET_ROWS:
        q, a = all_pairs[idx % len(all_pairs)]
        variants = paraphrase_question(q, a)
        if variants:
            # Выбираем случайную вариацию
            chosen = random.choice(variants)
            dataset.append((chosen, a))
        idx += 1
        # Защита от бесконечного цикла (если вариантов нет)
        if idx > len(all_pairs) * 10 and len(dataset) == len(all_pairs):
            # Дублируем исходные вопросы как запасной вариант
            dataset.append((q, a))
            break

# Перемешиваем датасет, чтобы разные темы чередовались
random.shuffle(dataset)

In [ ]:
# Сохранение в CSV
output_file = "university.csv"
with open(output_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(["question", "answer"])
    writer.writerows(dataset)

print(f"Датасет сохранён в {output_file}")
print(f"Всего строк: {len(dataset)}")

Датасет сохранён в university.csv
Всего строк: 5000


In [ ]:
pip install pandas scikit-learn sentence-transformers torch faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv('university.csv')
print(df.head())          # проверьте названия колонок
print(df.columns)

# Предположим, колонки: 'question', 'answer'
df = df.dropna(subset=['question', 'answer'])
df['question'] = df['question'].str.lower().str.strip()
# Можно добавить очистку от знаков препинания

                                            question  \
0  Скажите, пожалуйста, какие направления бакалав...   
1  Скажите, пожалуйста, квалификация «Бухгалтер» ...   
2               Объясните, что означает специалитет?   
3  Укажите цену заочное обучение (с ДОТ) за семестр?   
4  Скажите, пожалуйста, что означает «база основн...   

                                              answer  
0  «Специальное (дефектологическое) образование» ...  
1  38.02.01 «Экономика и бухгалтерский учет». Фор...  
2  Специалитет — традиционная для России ступень ...  
3  По предоставленной таблице, заочная форма с ДО...  
4  Это обучение на базе 9 классов (общее образова...  
Index(['question', 'answer'], dtype='object')


In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

In [ ]:
# Загрузка датасета
df = pd.read_csv("university.csv")
df = df[['question', 'answer']].dropna()
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df)
})